In [1]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
import re
from functools import reduce
from operator import add

In [2]:
from pyspark.sql import SparkSession
from src.utils.logger import get_logger
import src.utils.config as config 


print(f"DEBUG: Access Key is {config.MINIO_ACCESS_KEY[:3]} + ***") 
print(f"DEBUG: ENDPOINT is {config.MINIO_ENDPOINT}")

# Get the container's hostname dynamically

logger = get_logger(__name__)

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a configured Spark session for MinIO.
    """
    logger.info(f"Creating Spark Session: {app_name}")
    
    # This pulls the necessary S3A connectors from Maven Central
    
    

    spark = (   
        SparkSession.builder
        .appName(app_name)
        .master("spark://spark-master:7077")
        .config("spark.driver.host", config.DRIVER_HOST)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.port", config.SPARK_DRIVER_PORT)
        .config("spark.driver.blockManager.port", config.SPARK_BLOCK_MANAGER_PORT)
        .config("spark.sql.shuffle.partitions", "50")
        .config("spark.executor.instances", "1") # adjust based on resources
        .config("spark.executor.cores", "2")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
        #.config("spark.executor.memory", "2g") # adjust based on resources
        #.config("spark.driver.memory", "2g") # adjust based on resources

         # hadoop S3A Configuration
      
        .config("spark.hadoop.fs.s3a.endpoint", config.MINIO_ENDPOINT)
        .config("spark.hadoop.fs.s3a.access.key", config.MINIO_ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", config.MINIO_SECRET_KEY)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(config.MINIO_SECURE).lower())
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # prevent class resolution
        .config("spark.sql.caseSensitive", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.cores.max", "2")
        .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.hadoop.fs.s3a.fast.upload", "true") #performance improvement
        .config("spark.sql.files.maxPartitionBytes", "16777216") #16MB partitions
        #.config("spark.network.timeout", "1200s")
        #.config("spark.rpc.askTimeout", "600s")
        #.config("spark.executor.heartbeatInterval", "120s")
        #.config("spark.hadoop.fs.s3a.connection.timeout", "600000")
        #.config("spark.hadoop.fs.s3a.paging.maximum", "1000")
        
        .getOrCreate()
    )
    # Suppress verbose logs
    spark.sparkContext.setLogLevel("WARN")
    logger.info("Spark session created successfully")
    return spark

[CONFIG] Stage: transform
[CONFIG] Loaded env: /opt/spark-app/env/.env.transform
[CONFIG] MinIO Endpoint: lakehouse-minio:9000
[CONFIG] Access Key (masked): tra***
DEBUG: Access Key is tra + ***
DEBUG: ENDPOINT is lakehouse-minio:9000


In [3]:
spark= create_spark_session("notebook_test")
ireland_23= spark.read.parquet(
    "s3a://bronze/IRELAND/2023_BRONZE/")

2026-07-25 14:16:06 | INFO | lakehouse.__main__ | Creating Spark Session: notebook_test


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/25 14:17:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-07-25 14:17:51 | INFO | lakehouse.__main__ | Spark session created successfully


26/07/25 14:18:14 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
ireland_23

DataFrame[name_of_the_beneficiary_legal_entity_association: string, surname_of_beneficiary: string, if_belonging_to_a_group,_name_of_the_parent_entity_and_vat_or_tax_identification_number: string, municipality: string, code_of_measure_type_of_intervention_sector_as_set_in_annex_ix: string, specific_objective_â¹: string, start_date_â²: string, end_date_â³: string, amount_by_operation_under_eagf: string, total_of_eagf_amount_for_that_beneficiary: string, amount_by_operation_under_eafrd: string, total_of_eafrd_amount_for_that_beneficiary: string, amount_by_operation_under_co_financing: string, total_of_co_financed_amount_for_that_beneficiary: string, total_of_eafrd_and_co_financed_amounts: string, total_of_the_eu_amount_for_that_beneficiary: string, source_country: string, source_year: string, ingested_at: timestamp]

In [5]:
ireland_23.rdd.getNumPartitions()

2

In [5]:
ireland_23.printSchema()

root
 |-- name_of_the_beneficiary_legal_entity_association: string (nullable = true)
 |-- surname_of_beneficiary: string (nullable = true)
 |-- if_belonging_to_a_group,_name_of_the_parent_entity_and_vat_or_tax_identification_number: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- code_of_measure_type_of_intervention_sector_as_set_in_annex_ix: string (nullable = true)
 |-- specific_objective_â¹: string (nullable = true)
 |-- start_date_â²: string (nullable = true)
 |-- end_date_â³: string (nullable = true)
 |-- amount_by_operation_under_eagf: string (nullable = true)
 |-- total_of_eagf_amount_for_that_beneficiary: string (nullable = true)
 |-- amount_by_operation_under_eafrd: string (nullable = true)
 |-- total_of_eafrd_amount_for_that_beneficiary: string (nullable = true)
 |-- amount_by_operation_under_co_financing: string (nullable = true)
 |-- total_of_co_financed_amount_for_that_beneficiary: string (nullable = true)
 |-- total_of_eafrd_and_co_financed_amoun

In [6]:
for i , col in enumerate(ireland_23.columns, 1):
    print(f"{i:3d}| {col}")

  1| name_of_the_beneficiary_legal_entity_association
  2| surname_of_beneficiary
  3| if_belonging_to_a_group,_name_of_the_parent_entity_and_vat_or_tax_identification_number
  4| municipality
  5| code_of_measure_type_of_intervention_sector_as_set_in_annex_ix
  6| specific_objective_â¹
  7| start_date_â²
  8| end_date_â³
  9| amount_by_operation_under_eagf
 10| total_of_eagf_amount_for_that_beneficiary
 11| amount_by_operation_under_eafrd
 12| total_of_eafrd_amount_for_that_beneficiary
 13| amount_by_operation_under_co_financing
 14| total_of_co_financed_amount_for_that_beneficiary
 15| total_of_eafrd_and_co_financed_amounts
 16| total_of_the_eu_amount_for_that_beneficiary
 17| source_country
 18| source_year
 19| ingested_at


In [7]:

# 1. Build expressions dynamically based on each column's specific data type
count_expressions = []

for col_name, col_type in ireland_23.dtypes:
    # Base condition: works for ALL data types (Timestamps, Strings, Ints, etc.)
    condition = F.col(col_name).isNotNull()
    
    # Only append the NaN check if the column is a floating-point numeric type
    if col_type in ("double", "float"):
        condition = condition & ~F.isnan(F.col(col_name))

    # Rule B: Catch empty text cells in string columns
    elif col_type == "string":
        condition = condition & ~(F.trim(F.col(col_name))).isin("", "N/A", "n/a", "NA", "na")
        
    # Aggregate using the safe conditional block
    count_expressions.append(F.count(F.when(condition, 1)).alias(col_name))

# 2. Run the single, optimized aggregation across the cluster
counts_row = ireland_2023.select(count_expressions).first()

# Print your results
print(counts_row)

Row(name_of_the_beneficiary_legal_entity_association=299055, surname_of_beneficiary=16, if_belonging_to_a_group,_name_of_the_parent_entity_and_vat_or_tax_identification_number=5, municipality=298992, code_of_measure_type_of_intervention_sector_as_set_in_annex_ix=298992, specific_objective_â¹=20, start_date_â²=25, end_date_â³=30, amount_by_operation_under_eagf=122344, total_of_eagf_amount_for_that_beneficiary=0, amount_by_operation_under_eafrd=176643, total_of_eafrd_amount_for_that_beneficiary=15, amount_by_operation_under_co_financing=10, total_of_co_financed_amount_for_that_beneficiary=15, total_of_eafrd_and_co_financed_amounts=15, total_of_the_eu_amount_for_that_beneficiary=15, source_country=424549, source_year=424549, ingested_at=424549)


In [8]:
for column, valid_count in counts_row.asDict().items():
    print(f"{column:<65} | Valid Rows: {valid_count:,}")

name_of_the_beneficiary_legal_entity_association                  | Valid Rows: 299,055
surname_of_beneficiary                                            | Valid Rows: 16
if_belonging_to_a_group,_name_of_the_parent_entity_and_vat_or_tax_identification_number | Valid Rows: 5
municipality                                                      | Valid Rows: 298,992
code_of_measure_type_of_intervention_sector_as_set_in_annex_ix    | Valid Rows: 298,992
specific_objective_â¹                                             | Valid Rows: 20
start_date_â²                                                     | Valid Rows: 25
end_date_â³                                                       | Valid Rows: 30
amount_by_operation_under_eagf                                    | Valid Rows: 122,344
total_of_eagf_amount_for_that_beneficiary                         | Valid Rows: 0
amount_by_operation_under_eafrd                                   | Valid Rows: 176,643
total_of_eafrd_amount_for_that_beneficiary

In [9]:
total_rows = ireland_23.count()

print(f'{"Column Name": <65} |{"Missing Percentage"}')
print("-" *60)

for column, valid_count in counts_row.asDict().items():
    missing_percentage= ((total_rows - valid_count) / total_rows) *  100
    print(f"{column: <65} | {missing_percentage:.3f}%")

Column Name                                                       |Missing Percentage
------------------------------------------------------------
name_of_the_beneficiary_legal_entity_association                  | 29.559%
surname_of_beneficiary                                            | 99.996%
if_belonging_to_a_group,_name_of_the_parent_entity_and_vat_or_tax_identification_number | 99.999%
municipality                                                      | 29.574%
code_of_measure_type_of_intervention_sector_as_set_in_annex_ix    | 29.574%
specific_objective_â¹                                             | 99.995%
start_date_â²                                                     | 99.994%
end_date_â³                                                       | 99.993%
amount_by_operation_under_eagf                                    | 71.183%
total_of_eagf_amount_for_that_beneficiary                         | 100.000%
amount_by_operation_under_eafrd                                   | 58

In [10]:
col_types= dict(ireland_23.dtypes)

stats_list= []

# Extract stats from data

for col_name, valid_count in counts_row.asDict().items():
    missing_count= total_rows - valid_count
    missing_percentage= round((missing_count/total_rows) *100, 2) 

    stats_list.append({
       'column_name': col_name,
       'missing_count': missing_count,
        'missing_percentage': missing_percentage,
       'Non-Null count': valid_count,
       'dtype': col_types.get(col_name)


    })


# sort missing_percentage in ascending order
missing_ireland_stats_2023= sorted(stats_list, key= lambda x: x['missing_percentage'])
missing_ireland_stats_2023

[{'column_name': 'source_country',
  'missing_count': 0,
  'missing_percentage': 0.0,
  'Non-Null count': 424549,
  'dtype': 'string'},
 {'column_name': 'source_year',
  'missing_count': 0,
  'missing_percentage': 0.0,
  'Non-Null count': 424549,
  'dtype': 'string'},
 {'column_name': 'ingested_at',
  'missing_count': 0,
  'missing_percentage': 0.0,
  'Non-Null count': 424549,
  'dtype': 'timestamp'},
 {'column_name': 'name_of_the_beneficiary_legal_entity_association',
  'missing_count': 125494,
  'missing_percentage': 29.56,
  'Non-Null count': 299055,
  'dtype': 'string'},
 {'column_name': 'municipality',
  'missing_count': 125557,
  'missing_percentage': 29.57,
  'Non-Null count': 298992,
  'dtype': 'string'},
 {'column_name': 'code_of_measure_type_of_intervention_sector_as_set_in_annex_ix',
  'missing_count': 125557,
  'missing_percentage': 29.57,
  'Non-Null count': 298992,
  'dtype': 'string'},
 {'column_name': 'amount_by_operation_under_eafrd',
  'missing_count': 247906,
  'miss

In [11]:
# build a single list of distinct count expressions
distinct_exprs= [F.count_distinct(F.col(c)).alias(c) for c in ireland_23.columns]
distinct_row= ireland_23.select(distinct_exprs).first()

distinct_counts= distinct_row.asDict()

for col, count in distinct_counts.items():
    print(f"{col:<65} | Distinct Values: {count:,}")

name_of_the_beneficiary_legal_entity_association                  | Distinct Values: 92,181
surname_of_beneficiary                                            | Distinct Values: 9
if_belonging_to_a_group,_name_of_the_parent_entity_and_vat_or_tax_identification_number | Distinct Values: 2
municipality                                                      | Distinct Values: 29
code_of_measure_type_of_intervention_sector_as_set_in_annex_ix    | Distinct Values: 21
specific_objective_â¹                                             | Distinct Values: 8
start_date_â²                                                     | Distinct Values: 4
end_date_â³                                                       | Distinct Values: 7
amount_by_operation_under_eagf                                    | Distinct Values: 114,969
total_of_eagf_amount_for_that_beneficiary                         | Distinct Values: 1
amount_by_operation_under_eafrd                                   | Distinct Values: 64,520
tot

In [8]:
# we need to verify if te correspondin row in amount_by_operation_under_eagf is None or as a value
suspect_rows= ireland_23.filter(
    F.col("end_date_â³").rlike(r"^\d+(\.\d+)?$")
)
suspect_rows.select( 
    "start_date_â²",
    "end_date_â³",
    "amount_by_operation_under_eagf"
    
).show(5, truncate=False)

+-------------+-----------+------------------------------+
|start_date_â²|end_date_â³|amount_by_operation_under_eagf|
+-------------+-----------+------------------------------+
|NULL         |1081800.09 |NULL                          |
|NULL         |894754.33  |NULL                          |
|NULL         |356772.27  |NULL                          |
|NULL         |311891.08  |NULL                          |
|NULL         |571552.68  |NULL                          |
+-------------+-----------+------------------------------+



### let's fix the data misalignment issue

In [9]:


# 1. Define the condition expression
is_shifted = (
    F.col("end_date_â³").rlike(r"^\d+(\.\d+)?$") & 
    ((F.col("amount_by_operation_under_eagf") == 0.0) | F.col("amount_by_operation_under_eagf").isNull())
)

# 2. BAKE the condition into a temporary column first so it can't change
ireland_23_baked = ireland_23.withColumn("is_shifted_flag", is_shifted)

# 3. Apply transformations using the static flag column
ireland_23_fixed = ireland_23_baked \
    .withColumn(
        "amount_by_operation_under_eagf", 
        F.when(F.col("is_shifted_flag") == True, F.col("end_date_â³").cast("double"))
         .otherwise(F.col("amount_by_operation_under_eagf"))
    ) \
    .withColumn(
        "end_date_â³", 
        F.when(F.col("is_shifted_flag") == True, F.lit(None))
         .otherwise(F.col("end_date_â³"))
    ) \
    .drop("is_shifted_flag") # Clean up the temporary column

# 4. Verification
# We handle both true Nulls and literal "None" strings just in case
ireland_23_fixed.filter(
    (F.col("amount_by_operation_under_eagf") > 0.0) & 
    (F.col("end_date_â³").isNull() | (F.col("end_date_â³") == "None"))
).select(
    "start_date_â²",
    "end_date_â³",
    "amount_by_operation_under_eagf"
).show(5, truncate=False)

+-------------+-----------+------------------------------+
|start_date_â²|end_date_â³|amount_by_operation_under_eagf|
+-------------+-----------+------------------------------+
|NULL         |NULL       |1081800.09                    |
|NULL         |NULL       |894754.33                     |
|NULL         |NULL       |356772.27                     |
|NULL         |NULL       |311891.08                     |
|NULL         |NULL       |571552.68                     |
+-------------+-----------+------------------------------+



In [10]:
ireland_23_cleaned= ireland_23_fixed.select(
    F.col("name_of_the_beneficiary_legal_entity_association").alias("beneficiary"),
    F.col("municipality").alias("municipality"),
    F.col("source_country").alias("country"),
    F.col("source_year").alias("year"),
    
    F.col("code_of_measure_type_of_intervention_sector_as_set_in_annex_ix").alias("intervention_code"),
    F.col("amount_by_operation_under_eagf").cast(DoubleType()).alias("total_eagf_income_support"),
    F.col("amount_by_operation_under_eafrd").cast(DoubleType()).alias("total_eafrd_income_support"),
    F.col("amount_by_operation_under_co_financing").cast(DoubleType()).alias("national_cofunding_amount")
).fillna(0.0, subset=["total_eagf_income_support", "total_eafrd_income_support", "national_cofunding_amount"])

# fill nulls in text fields
ireland23_cleaned= ireland_23_cleaned.fillna("UNKNOWN", subset= ["beneficiary", "municipality", "intervention_code"])

In [11]:
ireland23_cleaned.show(5, truncate=False)

+---------------+------------+-------+----+-----------------------------------------+-------------------------+--------------------------+-------------------------+
|beneficiary    |municipality|country|year|intervention_code                        |total_eagf_income_support|total_eafrd_income_support|national_cofunding_amount|
+---------------+------------+-------+----+-----------------------------------------+-------------------------+--------------------------+-------------------------+
|UNKNOWN        |UNKNOWN     |IRELAND|2023|UNKNOWN                                  |0.0                      |0.0                       |0.0                      |
|DAVID KENNEDY  |CAVAN       |IRELAND|2023|Direct Payments                          |5092.43                  |0.0                       |0.0                      |
|DAVID KENNEDY  |CAVAN       |IRELAND|2023|Areas facing Natural/Specific Constraints|0.0                      |427.64                    |0.0                      |
|UNKNOWN  

In [12]:
spark.stop()